# Internal Dependencies
<br>  

### References
- [Analyze java package metrics in a graph database](https://joht.github.io/johtizen/data/2023/04/21/java-package-metrics-analysis.html)
- [Calculate metrics](https://101.jqassistant.org/calculate-metrics/index.html)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plot
from neo4j import GraphDatabase

In [2]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell 
# before starting jupyter notebook to provide the password for the user "neo4j". 
# It is not recommended to hardcode the password into jupyter notebook for security reasons.

driver = GraphDatabase.driver(uri="bolt://localhost:7687", auth=("neo4j", os.environ.get("NEO4J_INITIAL_PASSWORD")))
driver.verify_connectivity()

In [3]:
def get_cypher_query_from_file(cypher_file_name : str):
    with open(cypher_file_name) as file:
        return ' '.join(file.readlines())


def query_cypher_to_data_frame(filename : str, limit: int = -1):
    """
    Execute the Cypher query of the given file and returns the result.
    filename : str : The name of the file containing the Cypher query
    limit : int : The optional limit of rows to optimize the query. Default = -1 = no limit
    """
    cypher_query = get_cypher_query_from_file(filename)
    if limit > 0:
        cypher_query = "{query}\nLIMIT {row_limit}".format(query = cypher_query, row_limit = limit)
    records, summary, keys = driver.execute_query(cypher_query)
    return pd.DataFrame([r.values() for r in records], columns=keys)


def query_first_non_empty_cypher_to_data_frame(*filenames : str, limit: int = -1):
    """
    Executes the Cypher queries of the given files and returns the first result that is not empty.
    If all given file names result in empty results, the last (empty) result will be returned.
    By additionally specifying "limit=" the "LIMIT" keyword will appended to query so that only the first results get returned.
    """    
    result=pd.DataFrame()
    for filename in filenames:
        result=query_cypher_to_data_frame(filename, limit)
        if not result.empty:
            return result
    return result

In [4]:
#The following cell uses the build-in %html "magic" to override the CSS style for tables to a much smaller size.
#This is especially needed for PDF export of tables with multiple columns.

In [5]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

In [6]:
# Pandas DataFrame Display Configuration
pd.set_option('display.max_colwidth', 300)

## 1 - Modules

List the modules this notebook is based on. Different sorting variations help finding modules by their features and support larger code bases where the list of all modules gets very long.

Only the top 30 entries are shown. The whole table can be found in the following CSV report:  
`List_all_Typescript_modules`

In [7]:
internalModules = query_cypher_to_data_frame("../cypher/Internal_Dependencies/List_all_Typescript_modules.cypher")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: numberOfGitCommits)} {position: line: 9, column: 31, offset: 594} for query: '// List all existing internal Typescript modules. Requires "Set_localRootPath_for_modules.cypher", "Set_number_of...commits.cypher".\n \n MATCH (internalModule:TS:Module)-[:EXPORTS]->(internalElement:TS)\n  WITH internalModule.name                            AS moduleName\n      ,internalModule.rootProjectName                 AS rootProjectName\n      ,internalModule.localModuleDir                  AS localModuleDir\n      ,internalModule.incomingDependencies            AS inc

### Table 1a - Top 30 modules with the highest element count

In [8]:
# Sort by number of modules descending
internalModules.sort_values(by=['numberOfElements','moduleName'], ascending=[False, True]).reset_index(drop=True).head(30)

,rootProjectName,moduleName,numberOfElements,numberOfGitCommits,incomingDependencies,outgoingDependencies
0,react-router-7.13.0,react-router,169,0,364,13
1,react-router-7.13.0,utils,85,0,166,0
2,react-router-7.13.0,react-router,39,0,110,4
3,react-router-7.13.0,router,33,0,52,1
4,react-router-7.13.0,utils,28,0,36,29
5,react-router-7.13.0,context,27,0,25,12
6,react-router-7.13.0,history,25,0,37,0
7,react-router-7.13.0,utils,24,0,46,0
8,react-router-7.13.0,routeModules,21,0,27,6
9,react-router-7.13.0,server,21,0,38,19


### Table 1b - Top 30 modules with the highest number of incoming dependencies

The following table lists the top 30 internal modules that are used the most by other modules (highest count of incoming dependencies, highest in-degree).

In [9]:
# Sort by number of incoming dependencies descending
internalModules.sort_values(by=['incomingDependencies','moduleName'], ascending=[False, True]).reset_index(drop=True).head(30)

,rootProjectName,moduleName,numberOfElements,numberOfGitCommits,incomingDependencies,outgoingDependencies
0,react-router-7.13.0,react-router,169,0,364,13
1,react-router-7.13.0,utils,85,0,166,0
2,react-router-7.13.0,react-router,39,0,110,4
3,react-router-7.13.0,router,33,0,52,1
4,react-router-7.13.0,utils,24,0,46,0
5,react-router-7.13.0,server,21,0,38,19
6,react-router-7.13.0,history,25,0,37,0
7,react-router-7.13.0,utils,28,0,36,29
8,react-router-7.13.0,routeModules,21,0,27,6
9,react-router-7.13.0,context,27,0,25,12


### Table 1c - Top 30 modules with the highest number of outgoing dependencies

The following table lists the top 30 internal modules that are depending on the highest number of other modules (highest count of outgoing dependencies, highest out-degree).

In [10]:
# Sort by number of outgoing dependencies descending
internalModules.sort_values(by=['outgoingDependencies','moduleName'], ascending=[False, True]).reset_index(drop=True).head(30)

,rootProjectName,moduleName,numberOfElements,numberOfGitCommits,incomingDependencies,outgoingDependencies
0,react-router-7.13.0,route-chunks,12,0,8,310
1,react-router-7.13.0,plugin,17,0,9,150
2,react-router-7.13.0,plugin,1,0,1,54
3,react-router-7.13.0,config,12,0,16,37
4,react-router-7.13.0,copy-template,2,0,2,33
5,react-router-7.13.0,remove-exports,1,0,2,30
6,react-router-7.13.0,utils,28,0,36,29
7,react-router-7.13.0,with-props,1,0,1,29
8,react-router-7.13.0,create-react-router,2,0,0,28
9,react-router-7.13.0,routes,15,0,25,28


### Table 1d - Top 30 modules with the lowest element count

In [11]:
# Sort by number of elements ascending
internalModules.sort_values(by=['numberOfElements','moduleName'], ascending=[True, True]).reset_index(drop=True).head(30)

,rootProjectName,moduleName,numberOfElements,numberOfGitCommits,incomingDependencies,outgoingDependencies
0,react-router-7.13.0,actions,1,0,3,0
1,react-router-7.13.0,arcTableSessionStorage,1,0,1,6
2,react-router-7.13.0,binaryTypes,1,0,1,0
3,react-router-7.13.0,browser,1,0,1,0
4,react-router-7.13.0,cloudflare,1,0,0,0
5,react-router-7.13.0,cloudflare-dev-proxy,1,0,1,15
6,react-router-7.13.0,combine-urls,1,0,1,0
7,react-router-7.13.0,cookieStorage,1,0,2,2
8,react-router-7.13.0,data,1,0,0,0
9,react-router-7.13.0,detectPackageManager,1,0,1,1


### Table 1e - Top 30 modules with the lowest number of incoming dependencies

The following table lists the top 30 internal modules that are used the least by other modules (lowest count of incoming dependencies, lowest in-degree).

In [12]:
# Sort by number of incoming dependencies ascending
internalModules.sort_values(by=['incomingDependencies','moduleName'], ascending=[True, True]).reset_index(drop=True).head(30)

,rootProjectName,moduleName,numberOfElements,numberOfGitCommits,incomingDependencies,outgoingDependencies
0,react-router-7.13.0,cloudflare,1,0,0,0
1,react-router-7.13.0,create-react-router,2,0,0,28
2,react-router-7.13.0,data,1,0,0,0
3,react-router-7.13.0,global,2,0,0,0
4,react-router-7.13.0,internal,2,0,0,0
5,react-router-7.13.0,invariant,1,0,0,0
6,react-router-7.13.0,invariant,1,0,0,0
7,react-router-7.13.0,invariant,1,0,0,0
8,react-router-7.13.0,links,8,0,0,9
9,react-router-7.13.0,params,1,0,0,0


### Table 1f - Top 30 modules with the lowest number of outgoing dependencies

The following table lists the top 30 internal modules that are depending on the lowest number of other modules (lowest count of outgoing dependencies, lowest out-degree).

In [13]:
# Sort by number of outgoing dependencies ascending
internalModules.sort_values(by=['outgoingDependencies','moduleName'], ascending=[True, True]).reset_index(drop=True).head(30)

,rootProjectName,moduleName,numberOfElements,numberOfGitCommits,incomingDependencies,outgoingDependencies
0,react-router-7.13.0,actions,1,0,3,0
1,react-router-7.13.0,binaryTypes,1,0,1,0
2,react-router-7.13.0,browser,1,0,1,0
3,react-router-7.13.0,build,5,0,11,0
4,react-router-7.13.0,cache,2,0,3,0
5,react-router-7.13.0,cloudflare,1,0,0,0
6,react-router-7.13.0,combine-urls,1,0,1,0
7,react-router-7.13.0,config,4,0,5,0
8,react-router-7.13.0,context,2,0,3,0
9,react-router-7.13.0,crypto,2,0,2,0


## 2 - Cyclic Dependencies

Cyclic dependencies occur when one module uses an elements of another module and vice versa. 
These dependencies can lead to problems when one of these modules needs to be changed.

### Table 2a - Cyclic Dependencies Overview

Show the top 40 cyclic dependencies sorted by the most promising to resolve first. This is done by calculating the number of forward dependencies (first cycle participant to second cycle participant) in relation to backward dependencies (second cycle participant back to first cycle participant). The higher this rate (approaching 1), the easier it should be to resolve the cycle by focussing on the few backward dependencies.

Only the top 40 entries are shown. The whole table can be found in the following CSV report:  
`Cyclic_Dependencies_for_Typescript`

**Columns:**
- *projectFileName* identifies the project of the first participant of the cycle
- *modulePathName* identifies the module of the first participant of the cycle
- *dependentProjectFileName* identifies the project of the second participant of the cycle
- *dependentModulePathName* identifies the module of the second participant of the cycle
- *forwardToBackwardBalance* is between 0 and 1. High for many forward and few backward dependencies.
- *numberForward* contains the number of dependencies from the first participant of the cycle to the second one
- *numberBackward* contains the number of dependencies from the second participant of the cycle back to the first one
- *someForwardDependencies* lists some forward dependencies in the text format "type1 -> type2"
- *backwardDependencies* lists the backward dependencies in the format "type1 <- type2" that are recommended to get resolved

In [14]:
cyclic_dependencies = query_cypher_to_data_frame("../cypher/Cyclic_Dependencies/Cyclic_Dependencies_for_Typescript.cypher")
cyclic_dependencies.head(40)

,projectFileName,moduleName,dependentProjectFileName,dependentModulePathName,forwardToBackwardBalance,numberForward,numberBackward,forwardDependencyExamples,backwardDependencyExamples
0,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,0.794872,35,4,"[RouterContextProvider->RouterContext, createContext->RouterContext, createStaticHandler->ResultType, createStaticHandler->isRouteErrorResponse, createStaticHandler->DataStrategyMatch, matchRoutes->AgnosticRouteObject, createStaticHandler->AgnosticRouteObject, matchRSCServerRequest->stripBasenam...","[MiddlewareNextFunction<-MiddlewareFunction, data<-convertRouteMatchToUiMatch, RouterContext<-RouterContextProvider, RouterContext<-createContext]"
1,react-router,./lib/router/router.ts,react-router,./lib/router/instrumentation.ts,0.750000,7,1,"[createRouter->getRouteInstrumentationUpdates, createStaticHandler->getRouteInstrumentationUpdates, createRouter->unstable_InstrumentRouteFunction, createStaticHandler->unstable_InstrumentRouteFunction, createRouter->unstable_InstrumentRouterFunction, createRouter->instrumentClientSideRouter, Ro...",[Router<-instrumentClientSideRouter]
2,react-router,./lib/server-runtime/routes.ts,react-router,./index.ts,0.714286,6,1,"[createStaticHandlerDataRoutes->replace, createStaticHandlerDataRoutes->SingleFetchRedirectSymbol, createStaticHandlerDataRoutes->decodeViaTurboStream, createStaticHandlerDataRoutes->MiddlewareFunction, createStaticHandlerDataRoutes->redirect, createStaticHandlerDataRoutes->redirectDocument]",[ServerRouteManifest<-ServerBuild]
3,react-router,./lib/server-runtime/data.ts,react-router,./index.ts,0.666667,5,1,"[callRouteHandler->LoaderFunction, callRouteHandler->ActionFunctionArgs, callRouteHandler->ActionFunction, callRouteHandler->LoaderFunctionArgs, callRouteHandler->DataWithResponseInit]",[AppLoadContext<-RequestHandler]
4,react-router,./lib/context.ts,react-router,./lib/router/router.ts,0.600000,4,1,"[DataRouterContextObject->Router, DataRouterStateContext->Router, DataRouterContextObject->StaticHandlerContext, DataRouterStateContext->RouterState]",[RouteObject<-createRouter]
5,react-router,./lib/server-runtime/sessions.ts,react-router,./index-react-server.ts,0.600000,12,3,"[isSession->IsSessionFunction, createSessionStorage->isCookie, createSessionStorage->SessionStorage, SessionIdStorageStrategy->CookieSignatureOptions, warnOnceAboutSigningSessionCookie->Cookie, createSessionStorage->Cookie, SessionIdStorageStrategy->Cookie, createSessionStorage->SessionIdStorage...","[createSessionStorage<-createMemorySessionStorage, createSession<-createCookieSessionStorage, warnOnceAboutSigningSessionCookie<-createCookieSessionStorage]"
6,react-router,./lib/server-runtime/sessions.ts,react-router,./index.ts,0.600000,12,3,"[warnOnceAboutSigningSessionCookie->Cookie, createSessionStorage->Cookie, SessionIdStorageStrategy->Cookie, isSession->IsSessionFunction, SessionIdStorageStrategy->CookieSignatureOptions, createSessionStorage->isCookie, createSessionStorage->SessionIdStorageStrategy, SessionStorage->Session, Cre...","[createSession<-createCookieSessionStorage, warnOnceAboutSigningSessionCookie<-createCookieSessionStorage, createSessionStorage<-createMemorySessionStorage]"
7,react-router,./lib/dom/ssr/fog-of-war.ts,react-router,./index.ts,0.555556,14,4,"[useFogOFWarDiscovery->RouteModules, fetchAndApplyManifestPatches->RouteModules, getPatchRoutesOnNavigationFunction->RouteModules, fetchAndApplyManifestPatches->createClientRoutes, getPartialManifest->AssetsManifest, getPatchRoutesOnNavigationFunction->AssetsManifest, useFogOFWarDiscovery->Asset...","[fetchAndApplyManifestPatches<-useFogOFWarDiscovery, isFogOfWarEnabled<-useFogOFWarDiscovery, fetchAndApplyManifestPatches<-getPatchRoutesOnNavigationFunction, isFogOfWarEnabled<-getPatchRoutesOnNavigationFunction]"
8,react-router-dev,./vite/plugin.ts,react-router-dev,./vite/styles.ts,0.500000,3,1,"[reactRouterVitePlugin->isCssModulesFile, reactRouterVitePlugin->getStylesFo

### Table 2b - Cyclic Dependencies Break Down

Lists modules with cyclic dependencies with every dependency in a separate row sorted by the most promising dependency first.

Only the top 40 entries are shown. The whole table can be found in the following CSV report:  
`Cyclic_Dependencies_Breakdown_for_Typescript`

**Columns in addition to Table 2a:**
- *dependency* shows the cycle dependency in the text format "type1 -> type2" (forward) or "type2<-type1" (backward)

In [15]:
cyclic_dependencies_breakdown = query_cypher_to_data_frame("../cypher/Cyclic_Dependencies/Cyclic_Dependencies_Breakdown_for_Typescript.cypher",limit=40)
cyclic_dependencies_breakdown

,projectFileName,moduleName,dependentProjectFileName,dependentModulePathName,dependency,forwardToBackwardBalance,numberForward,numberBackward
0,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,matchRoutes->matchRoutesImpl,0.794872,35,4
1,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,redirectDocument->redirectDocument,0.794872,35,4
2,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,createStaticHandler->convertRoutesToDataRoutes,0.794872,35,4
3,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,Await->TrackedPromise,0.794872,35,4
4,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,createStaticHandler->SuccessResult,0.794872,35,4
5,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,RSCRouteMatch->Params,0.794872,35,4
6,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,replace->replace,0.794872,35,4
7,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,data->DataWithResponseInit,0.794872,35,4
8,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,replace->RedirectFunction,0.794872,35,4
9,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,RouterContext<-createContext,0.794872,35,4


### Table 2c - Cyclic Dependencies Break Down - Backward Dependencies Only

Lists modules with cyclic dependencies with every dependency in a separate row sorted by the most promising  dependency first. This table only contains the backward dependencies from the second participant of the cycle back to the first one that are the most promising to resolve.

Only the top 40 entries are shown. The whole table can be found in the following CSV report:  
`Cyclic_Dependencies_Breakdown_BackwardOnly_for_Typescript`

In [16]:
cyclic_dependencies_breakdown_backward = query_cypher_to_data_frame("../cypher/Cyclic_Dependencies/Cyclic_Dependencies_Breakdown_Backward_Only_for_Typescript.cypher",limit=40)
cyclic_dependencies_breakdown_backward

,projectFileName,moduleName,dependentProjectFileName,dependentModulePathName,dependency,forwardToBackwardBalance,numberForward,numberBackward
0,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,RouterContext<-createContext,0.794872,35,4
1,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,data<-convertRouteMatchToUiMatch,0.794872,35,4
2,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,RouterContext<-RouterContextProvider,0.794872,35,4
3,react-router,./index-react-server.ts,react-router,./lib/router/utils.ts,MiddlewareNextFunction<-MiddlewareFunction,0.794872,35,4
4,react-router,./lib/router/router.ts,react-router,./lib/router/instrumentation.ts,Router<-instrumentClientSideRouter,0.750000,7,1
5,react-router,./lib/server-runtime/routes.ts,react-router,./index.ts,ServerRouteManifest<-ServerBuild,0.714286,6,1
6,react-router,./lib/server-runtime/data.ts,react-router,./index.ts,AppLoadContext<-RequestHandler,0.666667,5,1
7,react-router,./lib/context.ts,react-router,./lib/router/router.ts,RouteObject<-createRouter,0.600000,4,1
8,react-router,./lib/server-runtime/sessions.ts,react-router,./index-react-server.ts,warnOnceAboutSigningSessionCookie<-createCookieSessionStorage,0.600000,12,3
9,react-router,./lib/server-runtime/sessions.ts,react-router,./index.ts,createSessionStorage<-createMemorySessionStorage,0.600000,12,3


## 3 - Module Usage

### Table 3a - Elements that are used by multiple modules

This table shows the top 40 modules that are used by the highest number of different modules. The whole table can be found in the CSV report `WidelyUsedTypescriptElements`.


In [17]:
elements_used_by_many_modules=query_cypher_to_data_frame("../cypher/Internal_Dependencies/List_elements_that_are_used_by_many_different_modules_for_Typescript.cypher", limit=40)
elements_used_by_many_modules

,fullQualifiedDependentElementName,dependentElementModuleName,dependentElementName,dependentElementLabels,numberOfUsingModules
0,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/history.ts"".Location",history.ts,Location,Interface,9
1,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router-dev/config/routes.ts"".RouteManifestEntry",routes.ts,RouteManifestEntry,Interface,9
2,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router-dev/vite/vite.ts"".getVite",vite.ts,getVite,Function,9
3,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router-dev/vite/vite.ts"".preloadVite",vite.ts,preloadVite,Function,9
4,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/utils.ts"".AgnosticRouteMatch",utils.ts,AgnosticRouteMatch,Interface,7
5,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/dom/ssr/routeModules.ts"".RouteModules",routeModules.ts,RouteModules,Interface,7
6,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/router.ts"".StaticHandlerContext",router.ts,StaticHandlerContext,Interface,7
7,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/create-react-router/utils.ts"".color",utils.ts,color,Variable,7
8,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/utils.ts"".DataWithResponseInit",utils.ts,DataWithResponseInit,Class,6
9,"""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/utils.ts"".ErrorResponseImpl",utils.ts,ErrorResponseImpl,Class,6


### Table 3b - Elements that are used by multiple modules

This table shows the top 30 modules that only use a few (compared to all existing) elements of another module.
The whole table can be found in the CSV report `ModuleElementsUsageTypescript`.

In [18]:
used_packages_of_dependent_artifact=query_cypher_to_data_frame("../cypher/Internal_Dependencies/How_many_elements_compared_to_all_existing_are_used_by_dependent_modules_for_Typescript.cypher",limit=30)
used_packages_of_dependent_artifact

,sourceModuleName,dependentModuleName,dependentElementsCount,dependentModuleElementsCount,elementUsagePercentage,dependentElementFullNameExamples,dependentElementNameExamples
0,memoryStorage,react-router,1,176,0.005682,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/server-runtime/sessions"".SessionStorage]",[SessionStorage]
1,route-data,react-router,1,169,0.005917,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/server-runtime/data.ts"".AppLoadContext]",[AppLoadContext]
2,mode,react-router,1,169,0.005917,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/server-runtime/mode.ts"".ServerMode]",[ServerMode]
3,routeMatching,react-router,2,176,0.011364,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/utils.ts"".matchRoutes, ""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/r...","[matchRoutes, Params]"
4,entry,utils,1,85,0.011765,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/utils.ts"".RouteManifest]",[RouteManifest]
5,links,utils,1,85,0.011765,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/utils.ts"".AgnosticDataRouteMatch]",[AgnosticDataRouteMatch]
6,route-module-annotations,react-router,2,169,0.011834,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/links.ts"".LinkDescriptor, ""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/sourc...","[LinkDescriptor, MetaDescriptor]"
7,internal,react-router,2,169,0.011834,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/links.ts"".LinkDescriptor, ""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/sourc...","[LinkDescriptor, MetaDescriptor]"
8,route-modules,react-router,3,176,0.017045,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/rsc/server.rsc.ts"".RSCRenderPayload, ""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/s...","[RSCRenderPayload, RSCRouteManifest, RouteModules]"
9,entry,react-router,3,169,0.017751,"[""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13.0/source/react-router-7.13.0/packages/react-router/lib/router/router.ts"".StaticHandlerContext, ""/home/runner/work/code-graph-analysis-examples/code-graph-analysis-examples/temp/react-router-7.13....","[StaticHandlerContext, RouteModules, AssetsManifest]"


### Table 3c - Distance distribution between dependent files

This table shows the file directory distance distribution between dependent files. Intuitively, the distance is given by the fewest number of change directory commands needed to navigate between a file and a dependency it uses. Those are aggregate to see how many dependent files are in the same directory, how many are just one change directory command apart, and so on.

In [19]:
query_first_non_empty_cypher_to_data_frame("../cypher/Internal_Dependencies/Get_file_distance_as_shortest_contains_path_for_dependencies.cypher",
                                           "../cypher/Internal_Dependencies/Set_file_distance_as_shortest_contains_path_for_dependencies.cypher", limit=20)

,dependency.fileDistanceAsFewestChangeDirectoryCommands,numberOfDependencies,numberOfDependencyUsers,numberOfDependencyProviders,examples
0,0,148,69,86,"[./server.ts uses ./binaryTypes.ts, ./server.ts uses ./index.ts, ./index.ts uses ./server.ts, ./index.ts uses ./copy-template.ts]"
1,3,41,24,29,"[./index.ts uses ./sessions/arcTableSessionStorage.ts, ./lib/errors.ts uses ./index.ts, ./lib/context.ts uses ./index.ts, ./lib/server-runtime/server.ts uses ./lib/actions.ts]"
2,4,116,43,40,"[./lib/rsc/route-modules.ts uses ./dom-export.ts, ./lib/rsc/server.rsc.ts uses ./dom-export.ts, ./lib/rsc/server.rsc.ts uses ./index-react-server-client.ts, ./lib/types/route-module-annotations.ts uses ./index-react-server.ts]"
3,5,48,22,18,"[./lib/server-runtime/sessions/cookieStorage.ts uses ./index-react-server.ts, ./lib/dom/ssr/fog-of-war.ts uses ./index-react-server.ts, ./lib/server-runtime/sessions/memoryStorage.ts uses ./index-react-server.ts, ./lib/dom/ssr/routeModules.ts uses ./index-react-server.ts]"
4,6,1,1,1,[./lib/server-runtime/single-fetch.ts uses ./vendor/turbo-stream-v2/turbo-stream.ts]
